In [1]:
from utility.adjust_redshift import adjust_redshift

from sparcl.client import SparclClient

import numpy as np
import pandas as pd
import json
import h5py
import time

property_file_path = "data/target_properties.csv"
spectra_hdf5_path = "data/batches/spectra.h5"

In [38]:
df = pd.read_csv(property_file_path,index_col=0)
df.info()

<class 'pandas.core.frame.DataFrame'>
Index: 404142 entries, 0 to 404141
Data columns (total 10 columns):
 #   Column              Non-Null Count   Dtype  
---  ------              --------------   -----  
 0   targetid            404142 non-null  int64  
 1   z                   404142 non-null  float64
 2   zerr                404142 non-null  float64
 3   civ_1549_flux       404142 non-null  float64
 4   civ_1549_flux_ivar  404142 non-null  float64
 5   flux_z              404142 non-null  float64
 6   flux_ivar_z         404142 non-null  float64
 7   flux_w3             404142 non-null  float64
 8   flux_ivar_w3        404142 non-null  float64
 9   desiname            404142 non-null  object 
dtypes: float64(8), int64(1), object(1)
memory usage: 33.9+ MB


In [39]:
# store the duplicate targets
duplicate_targetidxs = []

In [45]:
len(duplicates)

11013

In [40]:
# # detect duplicates, add to list, then remove from dataframe
# duplicates = df[df["targetid"].duplicated()]["targetid"].unique()
# duplicate_targetidxs.append(duplicates)  # already done
# df = df.drop_duplicates(subset="targetid",keep=False)

In [5]:
# target_idxs = [int(x) for x in df["targetid"]]

# n = len(target_idxs)
# batch_size = 500 # sparcl cannot take more than 500 idxs at once 
# n_batches = int((n / batch_size)+1)

# print(f"{batch_size=}, {n_batches=}")

batch_size=500, n_batches=764


In [6]:
# client = SparclClient()

In [7]:
# output_fields = ['targetid', 'redshift', 'wavelength', 'flux', 'ivar']

# results = client.retrieve_by_specid(specid_list=test_idxs, include=output_fields, dataset_list=["DESI-DR1"],limit=None,fmt="pandas")

In [ ]:
# duplicates = results[results["targetid"].duplicated()]["targetid"].unique()
# duplicate_targetidxs.append(duplicates)
# results = results.drop_duplicates(subset="targetid",keep=False)

In [ ]:
# for idx in results.index:
#     lam, flux, ivar = adjust_redshift(results.loc[idx,"redshift"], results.loc[idx,"wavelength"], results.loc[idx,"flux"], results.loc[idx,"ivar"])

#     valid_pixel_mask = ((lam > 1150) & (lam < 1810)) & (ivar > 0) & np.isfinite(ivar) & (flux != 0)

#     arr = np.array([lam[valid_pixel_mask],flux[valid_pixel_mask], ivar[valid_pixel_mask]])

#     with h5py.File("spectra.h5", "a") as f: 
#         dset = f.create_dataset(  
#             f"{results.loc[idx,'targetid']}", data=arr  
#         )



In [12]:
for i in range(715,764):
    tic = time.time()
    print(f"Start batch {i}")

    # get next 500 targetids
    curr_idxs = target_idxs[batch_size*i:batch_size*(i+1)]

    # retrieve spectra
    results = client.retrieve_by_specid(specid_list=curr_idxs, include=output_fields, dataset_list=["DESI-DR1"],limit=None,fmt="pandas")

    # detect duplicates, add to duplicate list, then remove them
    duplicates = results[results["targetid"].duplicated()]["targetid"].unique()
    duplicate_targetidxs.extend(duplicates)
    results = results.drop_duplicates(subset="targetid",keep=False)

    # redshift correct and truncate each spectrum to relevant window, then save as (3,N) numpy array to HDF5 file
    for idx in results.index:
        lam, flux, ivar = adjust_redshift(results.loc[idx,"redshift"], results.loc[idx,"wavelength"], results.loc[idx,"flux"], results.loc[idx,"ivar"])

        # valid pixels are: lambda inside relevant range (outer continuum window borders), flux is nonzero, ivar is nonzero and finite
        # get rid of pixels with 0 and infinite ivar (PyQSOFit does that similarly), as well as zero flux pixels (both PyQSOFit and VANDELS paper by Saldana-Lopez also does that)
        valid_pixel_mask = ((lam > 1150) & (lam < 1810)) & (ivar > 0) & np.isfinite(ivar) & (flux != 0)

        arr = np.array([lam[valid_pixel_mask],flux[valid_pixel_mask], ivar[valid_pixel_mask]])

        with h5py.File(spectra_hdf5_path, "a") as f: 
            dset = f.create_dataset(  
                f"{results.loc[idx,'targetid']}", data=arr  
            )
    
    toc = time.time() - tic
    print(f"Batch {i} finished in {toc:.2f}s\n")


Start batch 715
Batch 715 finished in 710.90s

Start batch 716
Batch 716 finished in 77.79s

Start batch 717
Batch 717 finished in 119.01s

Start batch 718
Batch 718 finished in 57.33s

Start batch 719
Batch 719 finished in 42.72s

Start batch 720
Batch 720 finished in 73.51s

Start batch 721
Batch 721 finished in 40.48s

Start batch 722
Batch 722 finished in 64.43s

Start batch 723
Batch 723 finished in 117.58s

Start batch 724
Batch 724 finished in 26.94s

Start batch 725
Batch 725 finished in 308.19s

Start batch 726
Batch 726 finished in 80.22s

Start batch 727
Batch 727 finished in 111.01s

Start batch 728
Batch 728 finished in 55.48s

Start batch 729
Batch 729 finished in 111.60s

Start batch 730
Batch 730 finished in 49.62s

Start batch 731
Batch 731 finished in 60.05s

Start batch 732
Batch 732 finished in 76.44s

Start batch 733
Batch 733 finished in 66.28s

Start batch 734
Batch 734 finished in 56.68s

Start batch 735
Batch 735 finished in 89.82s

Start batch 736
Batch 736 fi

In [15]:
with h5py.File(spectra_hdf5_path, "r") as f:
    # test = f[f"{results.loc[400,'targetid']}"][()]
    print(len(set(f.keys())))

380510


In [17]:
n_non_unique = 12139

In [14]:
with open("data/batches/duplicate_targets.txt","a") as f:
    f.write(str(duplicate_targetidxs))

In [51]:
# duplicate_targetidxs